# 유튜버 이탈 예측 및 주요 원인 분석 모델 (Random Forest)

이 노트북은 전처리 산출물인 preprocessed_data/X.csv, y.csv를 기준으로 **Random Forest** 이탈 예측 모델을 구축합니다.

- 입력 데이터: notebooks/01_data_collection/EDA/preprocessed_data/X.csv, y.csv
- 핵심 피처 사용 및 결측치 대체는 Pipeline(SimpleImputer -> RandomForestClassifier) 내에서 처리합니다.

In [ ]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("shap") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap"])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import shap

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    log_loss,
)
import warnings

warnings.filterwarnings('ignore')
available_fonts = {font.name for font in fm.fontManager.ttflist}
if 'Malgun Gothic' in available_fonts:
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif 'AppleGothic' in available_fonts:
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 추가 모델 임포트
try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
# 새 EDA 전처리 산출물 로드
PREPROCESSED_DIR = '../../notebooks/01_data_collection/EDA/preprocessed_data'

X_all = pd.read_csv(f'{PREPROCESSED_DIR}/X.csv')
y_df = pd.read_csv(f'{PREPROCESSED_DIR}/y.csv')
column_info = pd.read_csv(f'{PREPROCESSED_DIR}/column_info.csv')

data = X_all.merge(y_df, on='channel_identifier', how='inner')

print(f"X_all shape: {X_all.shape}")
print(f"y shape: {y_df.shape}")
print(f"merged data shape: {data.shape}")
print()
print("타깃 분포:")
print(data['is_churned'].value_counts())
print(f"이탈 비율: {data['is_churned'].mean():.2%}")
print(f"다수 클래스 기준 baseline accuracy: {(1 - data['is_churned'].mean()):.2%}")


## 2. 데이터 전처리 및 라벨링

이번 데이터는 EDA 단계에서 이미 전처리된 X.csv, y.csv를 사용합니다.

- y.csv의 is_churned가 타깃입니다.
- channel_identifier는 식별자이므로 학습 피처에서 제외합니다.
- column_info.csv와 중요도 진단 결과를 기준으로 churn 핵심 피처만 선택합니다.


In [ ]:
print("column_info 요약:")
display_cols = ["column", "source", "role", "dtype", "null_count", "description"]
try:
    display(column_info[display_cols])
except NameError:
    print(column_info[display_cols].to_string(index=False))


## 3. 피처 엔지니어링

새 EDA 전처리 데이터에서는 업로드 공백/규칙성 계열이 가장 강한 churn 신호로 나타났습니다.

핵심 축:

1. 업로드 공백/규칙성: max_gap_days, gap_ratio, std_upload_interval_days, avg_upload_interval_days, regularity_score, cv, hiatus_count_30d
2. 채널 규모: video_count, subscriber_count, total_views, channel_age_days
3. 성과/반응: avg_normal_view, avg_comment_count, avg_like_count, avg_view_count, std_view_count

민감 키워드 계열과 대부분의 장르 OHE는 churn보다는 최종 risk 계산 또는 평판 리스크 축에 더 적합하므로 이번 XGB churn 모델에서는 제외합니다.


In [ ]:
CHURN_CORE_FEATURES = [
    # 업로드 공백/규칙성 핵심 피처
    "max_gap_days",
    "gap_ratio",
    "std_upload_interval_days",
    "avg_upload_interval_days",
    "regularity_score",
    "cv",
    "hiatus_count_30d",

    # 채널 규모/운영 기간
    "video_count",
    "subscriber_count",
    "total_views",
    "channel_age_days",

    # 성과/반응 보조 피처
    "avg_normal_view",
    "avg_comment_count",
    "avg_like_count",
    "avg_view_count",
    "std_view_count",
]

missing_features = [feature for feature in CHURN_CORE_FEATURES if feature not in data.columns]
if missing_features:
    raise ValueError(f"데이터에 없는 피처가 있습니다: {missing_features}")

feature_info = column_info[column_info["column"].isin(CHURN_CORE_FEATURES)].copy()
feature_info["selected_order"] = feature_info["column"].map(
    {feature: idx + 1 for idx, feature in enumerate(CHURN_CORE_FEATURES)}
)
feature_info = feature_info.sort_values("selected_order")

print(f"선택 피처 수: {len(CHURN_CORE_FEATURES)}")
try:
    display(feature_info[["selected_order", "column", "source", "description"]])
except NameError:
    print(feature_info[["selected_order", "column", "source", "description"]].to_string(index=False))


### 피처(Feature) 설명

#### 선택한 핵심 피처

**업로드 공백/규칙성**
- max_gap_days: 최대 업로드 공백
- gap_ratio: 긴 공백 비율
- std_upload_interval_days: 업로드 간격 표준편차
- avg_upload_interval_days: 평균 업로드 간격
- regularity_score: 업로드 규칙성 점수, 낮을수록 위험
- cv: 업로드 간격 변동계수
- hiatus_count_30d: 30일 이상 공백 횟수

**채널 규모**
- video_count
- subscriber_count
- total_views
- channel_age_days

**성과/반응**
- avg_normal_view
- avg_comment_count
- avg_like_count
- avg_view_count
- std_view_count

#### 제외한 피처

- channel_identifier: 식별자
- collected_video_count: 대부분 수집 정책의 흔적에 가까워 중요도 낮음
- sensitive_score, n_sensitive_videos, sensitive_video_ratio, cat_*: 평판 리스크 축에 더 적합
- 대부분의 genre_*: churn 인과보다는 장르별 운영 패턴 보조 신호라 이번 핵심 모델에서는 제외


In [ ]:
ALL_FEATURES = CHURN_CORE_FEATURES
merged_df = data[["channel_identifier", "is_churned"] + ALL_FEATURES].copy()

print(f"학습 데이터 행 수: {len(merged_df):,}")
print(f"학습 피처 수: {len(ALL_FEATURES)}")
print()
print("피처별 결측치:")
missing = merged_df[ALL_FEATURES].isnull().sum().sort_values(ascending=False)
print(missing[missing > 0] if (missing > 0).any() else "  없음")


In [ ]:
EXCLUDED_FEATURE_GROUPS = {
    "identifier": ["channel_identifier"],
    "low_churn_signal": ["collected_video_count"],
    "reputation_risk": [
        "sensitive_score",
        "n_sensitive_videos",
        "sensitive_video_ratio",
        "cat_politics",
        "cat_hate",
        "cat_aggro",
        "cat_adult_illegal",
    ],
    "genre_ohe": [column for column in data.columns if column.startswith("genre_")],
}

print("이번 XGB churn 학습에서 제외한 주요 피처 그룹:")
for group, features in EXCLUDED_FEATURE_GROUPS.items():
    existing = [feature for feature in features if feature in data.columns]
    print(f"- {group}: {len(existing)}개")
    print(existing)


In [ ]:
# 선택 피처의 타깃별 평균 차이를 확인합니다.
summary_by_target = merged_df.groupby("is_churned")[ALL_FEATURES].mean().T
summary_by_target.columns = ["active_mean", "churned_mean"]
summary_by_target["diff_churned_minus_active"] = (
    summary_by_target["churned_mean"] - summary_by_target["active_mean"]
)
summary_by_target = summary_by_target.loc[ALL_FEATURES]

try:
    display(summary_by_target.round(3))
except NameError:
    print(summary_by_target.round(3).to_string())


In [ ]:
# 최종 학습 테이블
print(merged_df.head())


In [ ]:
merged_df.head()

In [ ]:
X = merged_df[ALL_FEATURES].copy()
y = merged_df['is_churned'].copy()

print(f"Feature count: {len(ALL_FEATURES)}")
print(ALL_FEATURES)
print()
print("타깃 분포:")
print(y.value_counts(normalize=True).rename("ratio").round(4))
print()
print("결측치 현황:")
missing = X.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0] if (missing > 0).any() else "  없음")


In [ ]:
RANDOM_STATE = 42
THRESHOLD_GRID = np.arange(0.05, 0.951, 0.01)
THRESHOLD_OBJECTIVE = "f1"  # accuracy보다 이탈 클래스 탐지력을 우선합니다.
model_results = {}


def make_train_valid_test_split(test_size=0.2, valid_size=0.25):
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_train_full,
        y_train_full,
        test_size=valid_size,
        random_state=RANDOM_STATE,
        stratify=y_train_full,
    )
    return X_train, X_valid, X_test, y_train, y_valid, y_test


def score_classifier(model, X_data):
    return model.predict_proba(X_data)[:, 1]


def calculate_binary_metrics(y_true, y_score, threshold):
    y_pred = (y_score >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score),
        "pr_auc": average_precision_score(y_true, y_score),
        "log_loss": log_loss(y_true, y_score),
    }


def threshold_key(metrics):
    if THRESHOLD_OBJECTIVE == "accuracy":
        return (metrics["accuracy"], metrics["f1"], metrics["recall"], metrics["precision"])
    return (metrics["f1"], metrics["recall"], metrics["precision"], metrics["roc_auc"])


def find_best_threshold(y_true, y_score):
    best_metrics = None
    best_key = None

    for threshold in THRESHOLD_GRID:
        metrics = calculate_binary_metrics(y_true, y_score, threshold)
        key = threshold_key(metrics)
        if best_key is None or key > best_key:
            best_key = key
            best_metrics = metrics

    return best_metrics


def show_table(df):
    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


def tune_model_candidates(model_name, candidates, score_func):
    rows = []
    best = None

    for idx, model_candidate in enumerate(candidates, start=1):
        model_candidate.fit(X_train, y_train)
        valid_score = np.clip(score_func(model_candidate, X_valid), 0, 1)
        valid_metrics = find_best_threshold(y_valid, valid_score)
        row = {"candidate": idx, **valid_metrics}
        rows.append(row)

        key = threshold_key(valid_metrics)
        if best is None or key > best["key"]:
            best = {
                "key": key,
                "candidate": idx,
                "model": model_candidate,
                "threshold": valid_metrics["threshold"],
                "valid_metrics": valid_metrics,
                "score_func": score_func,
            }

    summary = pd.DataFrame(rows).sort_values(
        ["f1", "recall", "precision", "roc_auc"], ascending=False
    ).reset_index(drop=True)
    print(f"=== {model_name}: validation tuning summary ({THRESHOLD_OBJECTIVE}) ===")
    show_table(summary)

    return best


def evaluate_tuned_model(model_name, result):
    y_score = np.clip(result["score_func"](result["model"], X_test), 0, 1)
    threshold = result["threshold"]
    y_pred = (y_score >= threshold).astype(int)
    test_metrics = calculate_binary_metrics(y_test, y_score, threshold)

    print(f"=== {model_name}: test metrics ===")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("confusion_matrix [[TN, FP], [FN, TP]]")
    print(confusion_matrix(y_test, y_pred))
    print(pd.Series(test_metrics).to_string())

    model_results[model_name] = {
        "candidate": result["candidate"],
        "threshold": threshold,
        "valid_metrics": result["valid_metrics"],
        "test_metrics": test_metrics,
        "model": result["model"],
    }

    return y_pred, y_score, test_metrics


X_train, X_valid, X_test, y_train, y_valid, y_test = make_train_valid_test_split()
print(f"Train: {X_train.shape}, Valid: {X_valid.shape}, Test: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.2%}, Valid churn rate: {y_valid.mean():.2%}, Test churn rate: {y_test.mean():.2%}")

## 4. RandomForestClassifier 모델링

In [ ]:
# Random Forest Classifier 학습 파이프라인
def make_rf_pipeline(**params):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            **params,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )),
    ])

# Random Forest 하이퍼파라미터 튜닝 후보군 정의
rf_candidates = [
    make_rf_pipeline(n_estimators=300, max_depth=5, min_samples_split=5, min_samples_leaf=2),
    make_rf_pipeline(n_estimators=500, max_depth=6, min_samples_split=10, min_samples_leaf=4),
    make_rf_pipeline(n_estimators=300, max_depth=8, min_samples_split=5, min_samples_leaf=1),
    make_rf_pipeline(n_estimators=400, max_depth=7, min_samples_split=7, min_samples_leaf=3),
]

rf_result = tune_model_candidates("RandomForestClassifier", rf_candidates, score_classifier)
model = rf_result["model"]
rf_pred, rf_prob, rf_test_metrics = evaluate_tuned_model("RandomForestClassifier", rf_result)

### 이탈 확률 기반 상위 채널 조회

In [ ]:
if model is not None:
    test_result = merged_df.loc[X_test.index, ['channel_identifier']].copy()
    test_result['churn_prob'] = rf_prob
    test_result['predicted_churn'] = rf_pred
    test_result['is_churned'] = y_test.values
    test_result = test_result.sort_values('churn_prob', ascending=False).reset_index(drop=True)

    print("=== RandomForestClassifier test set: top 10 channels by churn probability ===")
    print(test_result.head(10).to_string(index=False))

    all_prob = model.predict_proba(X)[:, 1]
    channel_churn = merged_df[['channel_identifier']].copy()
    channel_churn['churn_prob'] = all_prob
    channel_churn['churn_prob_pct'] = (channel_churn['churn_prob'] * 100).round(2)
    channel_churn = channel_churn.sort_values('churn_prob', ascending=False).reset_index(drop=True)

    print()
    print("=== RandomForestClassifier all channels: top 10 by churn probability ===")
    print(channel_churn.head(10).to_string(index=False))

## 5. 원인 분석 (SHAP)

SHAP(SHapley Additive exPlanations)를 활용해 Random Forest 모델의 예측 근거와 피처별 기여도를 시각화합니다.

In [ ]:
if model is not None:
    # 파이프라인에서 imputer와 model 분리 추출
    imputer = model.named_steps["imputer"]
    rf_raw_model = model.named_steps["model"]
    
    # SHAP 분석을 위한 데이터 프레임 변환
    X_train_imp = pd.DataFrame(imputer.transform(X_train), columns=ALL_FEATURES)
    X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=ALL_FEATURES)
    
    explainer = shap.TreeExplainer(rf_raw_model)
    shap_values = explainer.shap_values(X_test_imp)
    
    # Random Forest의 이진 분류 결과물 SHAP 차원 보정
    if isinstance(shap_values, list):
        shap_values_to_plot = shap_values[1]
    else:
        shap_values_to_plot = shap_values

    # Summary Plot
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values_to_plot, X_test_imp, show=False)
    plt.title("Random Forest SHAP Summary Plot", fontsize=15)
    plt.tight_layout()
    plt.show()